
# AgroUTMACH PIX
## NB04 — Tracking, IDs temporales y trayectorias

En NB03 ya demostramos que el detector puede trabajar sobre video y contar **aves visibles por fotograma**.  
Ahora añadiremos seguimiento temporal.

### Objetivos
1. localizar automáticamente un tramo estable del clip de alta densidad;
2. comparar **ByteTrack** y **BoT-SORT** sobre el mismo fragmento;
3. usar un diagnóstico operativo de persistencia de IDs;
4. seleccionar el tracker más estable para esta demostración;
5. generar un video final con:
   - detecciones;
   - IDs temporales;
   - trayectorias de los tracks más persistentes;
   - número de aves visibles;
   - número de tracks activos.

> **Importante:** un ID de tracking no identifica biológicamente a un pollo. Es un identificador temporal mantenido por el algoritmo mientras puede asociar detecciones entre fotogramas.

**Entorno recomendado: GPU NVIDIA L4.**



## 0. Active GPU

En Google Colab:

**Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU**

Seleccione **L4** si está disponible.


In [ ]:

from google.colab import drive
drive.mount('/content/drive')


In [ ]:

!nvidia-smi
!pip -q install -U ultralytics


In [ ]:

from pathlib import Path
from collections import defaultdict
import os, shutil, json, time, math
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from ultralytics import YOLO

ROOT = Path("/content/drive/MyDrive/AgroUTMACH_PIX")

MODEL_DRIVE = ROOT / "03_MODELOS" / "AgroUTMACH_PIX_YOLO11s_PIO_best.pt"
VIDEO_DRIVE = ROOT / "04_VIDEOS_FERIA" / "PIO_alta_original_45s.mp4"
COUNT_CSV = ROOT / "05_RESULTADOS" / "NB03_VIDEO" / "NB03_conteo_alta_densidad_por_frame.csv"

VIDEOS_OUT = ROOT / "04_VIDEOS_FERIA"
RESULTS_OUT = ROOT / "05_RESULTADOS" / "NB04_TRACKING"

RESULTS_OUT.mkdir(parents=True, exist_ok=True)

assert MODEL_DRIVE.exists(), f"Falta modelo: {MODEL_DRIVE}"
assert VIDEO_DRIVE.exists(), f"Falta video: {VIDEO_DRIVE}"
assert COUNT_CSV.exists(), f"Falta CSV NB03: {COUNT_CSV}"
assert torch.cuda.is_available(), "No hay GPU activa."

print("GPU:", torch.cuda.get_device_name(0))
print("Modelo:", MODEL_DRIVE)
print("Video:", VIDEO_DRIVE)



## 1. Copiar modelo y clip al almacenamiento local

El procesamiento de cientos de objetos por fotograma es más fluido desde `/content` que leyendo continuamente desde Drive.


In [ ]:

MODEL_LOCAL = Path("/content/AgroUTMACH_PIX_YOLO11s_PIO_best.pt")
VIDEO_LOCAL = Path("/content/PIO_alta_original_45s.mp4")

if not MODEL_LOCAL.exists():
    shutil.copy2(MODEL_DRIVE, MODEL_LOCAL)

if not VIDEO_LOCAL.exists() or VIDEO_LOCAL.stat().st_size != VIDEO_DRIVE.stat().st_size:
    print("Copiando clip a /content ...")
    t0 = time.time()
    shutil.copy2(VIDEO_DRIVE, VIDEO_LOCAL)
    print(f"✓ Copiado en {(time.time()-t0):.1f} s")
else:
    print("↪ Clip local ya disponible.")

print(f"Video: {VIDEO_LOCAL.stat().st_size/1024**2:.1f} MB")
print(f"Modelo: {MODEL_LOCAL.stat().st_size/1024**2:.1f} MB")



## 2. Seleccionar automáticamente un tramo estable de 20 segundos

NB03 mostró un valor mínimo anómalo de detecciones en algún fotograma del clip.  
Para la feria conviene evitar transiciones o fotogramas atípicos.

Buscaremos una ventana de **20 segundos** con:
- conteo medio alto;
- poca variabilidad;
- sin caídas severas.

Esto no cambia el dataset ni el modelo; solo selecciona un fragmento visualmente estable para la demostración.


In [ ]:

df_count = pd.read_csv(COUNT_CSV)

cap = cv2.VideoCapture(str(VIDEO_LOCAL))
fps = cap.get(cv2.CAP_PROP_FPS)
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration = n_frames / fps
cap.release()

WINDOW_SECONDS = 20
window = max(1, int(round(WINDOW_SECONDS * fps)))

counts = df_count["visible_birds"].astype(float)
roll_mean = counts.rolling(window=window, min_periods=window).mean()
roll_std = counts.rolling(window=window, min_periods=window).std()

# Puntaje de estabilidad: prioriza conteo alto y penaliza variabilidad.
score = roll_mean - 0.75 * roll_std
best_end_idx = int(score.idxmax())
best_start_idx = max(0, best_end_idx - window + 1)

stable_start_sec = best_start_idx / fps
stable_end_sec = min(duration, (best_end_idx + 1) / fps)

stable = df_count.iloc[best_start_idx:best_end_idx+1]

print(f"FPS: {fps:.2f}")
print(f"Clip total: {duration:.2f} s")
print(f"Tramo estable: {stable_start_sec:.2f}–{stable_end_sec:.2f} s")
print(f"Conteo medio: {stable['visible_birds'].mean():.1f}")
print(f"Mediana: {stable['visible_birds'].median():.1f}")
print(f"Mínimo: {stable['visible_birds'].min():.0f}")
print(f"Máximo: {stable['visible_birds'].max():.0f}")
print(f"Desv. est.: {stable['visible_birds'].std():.1f}")

stable_info = {
    "start_seconds": float(stable_start_sec),
    "end_seconds": float(stable_end_sec),
    "duration_seconds": float(stable_end_sec - stable_start_sec),
    "mean_visible_birds": float(stable["visible_birds"].mean()),
    "median_visible_birds": float(stable["visible_birds"].median()),
    "min_visible_birds": int(stable["visible_birds"].min()),
    "max_visible_birds": int(stable["visible_birds"].max()),
    "std_visible_birds": float(stable["visible_birds"].std()),
}

with open(RESULTS_OUT / "NB04_tramo_estable.json", "w", encoding="utf-8") as f:
    json.dump(stable_info, f, indent=2, ensure_ascii=False)


In [ ]:

STABLE_CLIP = Path("/content/PIO_tracking_stable_20s.mp4")
STABLE_CLIP_DRIVE = VIDEOS_OUT / "PIO_tracking_estable_original_20s.mp4"

cmd = (
    f'ffmpeg -y -hide_banner -loglevel error '
    f'-ss {stable_start_sec:.3f} -i "{VIDEO_LOCAL}" '
    f'-t {WINDOW_SECONDS:.3f} '
    f'-c:v libx264 -preset veryfast -crf 20 -pix_fmt yuv420p -an "{STABLE_CLIP}"'
)
rc = os.system(cmd)
assert rc == 0 and STABLE_CLIP.exists(), "No se pudo crear el clip estable."

shutil.copy2(STABLE_CLIP, STABLE_CLIP_DRIVE)
print("✓ Clip estable guardado:", STABLE_CLIP_DRIVE)



## 3. Comparación operativa: ByteTrack vs BoT-SORT

No tenemos anotaciones de identidad individual, por lo que **no calcularemos MOTA, IDF1 ni HOTA**.

En su lugar, para elegir un tracker de demostración mediremos indicadores operativos:

- **IDs únicos generados**;
- **nuevos IDs por segundo**;
- **mediana de duración de los tracks**;
- **percentil 75 de duración**;
- **promedio de tracks activos por fotograma**.

Estos indicadores ayudan a detectar fragmentación excesiva de IDs, pero **no sustituyen una evaluación formal de tracking**.


In [ ]:

CONF = 0.25
IOU = 0.60
IMGSZ = 960
MAX_DET = 1000
COMPARE_SECONDS = 10

def evaluate_tracker(video_path, tracker_yaml, seconds=10):
    model = YOLO(str(MODEL_LOCAL))
    cap = cv2.VideoCapture(str(video_path))
    fps_local = cap.get(cv2.CAP_PROP_FPS)
    max_frames = int(seconds * fps_local)

    first_seen = {}
    last_seen = {}
    active_counts = []
    unique_ids = set()
    frame_idx = 0
    t0 = time.time()

    while frame_idx < max_frames:
        ok, frame = cap.read()
        if not ok:
            break

        result = model.track(
            source=frame,
            persist=True,
            tracker=tracker_yaml,
            imgsz=IMGSZ,
            conf=CONF,
            iou=IOU,
            max_det=MAX_DET,
            device=0,
            verbose=False
        )[0]

        ids = []
        if result.boxes is not None and result.boxes.id is not None:
            ids = result.boxes.id.int().cpu().tolist()

        active_counts.append(len(ids))

        for tid in ids:
            unique_ids.add(tid)
            if tid not in first_seen:
                first_seen[tid] = frame_idx
            last_seen[tid] = frame_idx

        frame_idx += 1

    cap.release()

    lifetimes = [
        (last_seen[tid] - first_seen[tid] + 1) / fps_local
        for tid in unique_ids
        if tid in last_seen
    ]

    elapsed = time.time() - t0

    out = {
        "tracker": tracker_yaml,
        "frames": int(frame_idx),
        "seconds_analyzed": float(frame_idx / fps_local),
        "unique_ids": int(len(unique_ids)),
        "new_ids_per_second": float(len(unique_ids) / max(frame_idx / fps_local, 1e-9)),
        "mean_active_tracks": float(np.mean(active_counts)) if active_counts else 0.0,
        "median_track_lifetime_s": float(np.median(lifetimes)) if lifetimes else 0.0,
        "p75_track_lifetime_s": float(np.percentile(lifetimes, 75)) if lifetimes else 0.0,
        "processing_seconds": float(elapsed),
        "processing_fps": float(frame_idx / elapsed) if elapsed > 0 else 0.0,
    }
    return out

byte_metrics = evaluate_tracker(STABLE_CLIP, "bytetrack.yaml", COMPARE_SECONDS)
bot_metrics = evaluate_tracker(STABLE_CLIP, "botsort.yaml", COMPARE_SECONDS)

comparison = pd.DataFrame([byte_metrics, bot_metrics])
comparison


In [ ]:

comparison.to_csv(RESULTS_OUT / "NB04_comparacion_trackers.csv", index=False)

# Heurística operativa:
# 1) preferir mayor mediana de persistencia;
# 2) si son muy similares, preferir menor generación de IDs por segundo.
med_byte = byte_metrics["median_track_lifetime_s"]
med_bot = bot_metrics["median_track_lifetime_s"]

if abs(med_byte - med_bot) > 0.10:
    chosen = byte_metrics if med_byte > med_bot else bot_metrics
else:
    chosen = (
        byte_metrics
        if byte_metrics["new_ids_per_second"] <= bot_metrics["new_ids_per_second"]
        else bot_metrics
    )

CHOSEN_TRACKER = chosen["tracker"]

print("TRACKER SELECCIONADO PARA LA DEMOSTRACIÓN:", CHOSEN_TRACKER)
print(json.dumps(chosen, indent=2, ensure_ascii=False))

with open(RESULTS_OUT / "NB04_tracker_seleccionado.json", "w", encoding="utf-8") as f:
    json.dump({
        "selection_type": "diagnostico_operativo_no_ground_truth",
        "selected_tracker": CHOSEN_TRACKER,
        "bytetrack": byte_metrics,
        "botsort": bot_metrics,
    }, f, indent=2, ensure_ascii=False)



## 4. Ejecutar tracking completo sobre los 20 segundos estables

Guardaremos un registro por detección:

`frame, time_seconds, track_id, x1, y1, x2, y2, confidence`

Después elegiremos los tracks más persistentes para dibujar trayectorias sin llenar toda la pantalla de líneas.


In [ ]:

track_model = YOLO(str(MODEL_LOCAL))
cap = cv2.VideoCapture(str(STABLE_CLIP))

fps_track = cap.get(cv2.CAP_PROP_FPS)
rows = []
frame_idx = 0
t0 = time.time()

while True:
    ok, frame = cap.read()
    if not ok:
        break

    result = track_model.track(
        source=frame,
        persist=True,
        tracker=CHOSEN_TRACKER,
        imgsz=IMGSZ,
        conf=CONF,
        iou=IOU,
        max_det=MAX_DET,
        device=0,
        verbose=False
    )[0]

    if (
        result.boxes is not None
        and len(result.boxes) > 0
        and result.boxes.id is not None
    ):
        xyxy = result.boxes.xyxy.cpu().numpy()
        ids = result.boxes.id.int().cpu().numpy()
        confs = result.boxes.conf.cpu().numpy()

        for box, tid, confv in zip(xyxy, ids, confs):
            x1, y1, x2, y2 = box.tolist()
            rows.append({
                "frame": int(frame_idx),
                "time_seconds": float(frame_idx / fps_track),
                "track_id": int(tid),
                "x1": float(x1),
                "y1": float(y1),
                "x2": float(x2),
                "y2": float(y2),
                "confidence": float(confv),
            })

    frame_idx += 1
    if frame_idx % max(1, int(fps_track * 5)) == 0:
        elapsed = time.time() - t0
        print(f"{frame_idx:4d} frames | {frame_idx/fps_track:5.1f}s | elapsed {elapsed/60:.1f} min")

cap.release()

tracks_df = pd.DataFrame(rows)
TRACK_LOG = RESULTS_OUT / "NB04_tracks_por_frame.csv"
tracks_df.to_csv(TRACK_LOG, index=False)

print("\n✓ Tracking completo")
print("Filas registradas:", f"{len(tracks_df):,}")
print("IDs únicos:", tracks_df["track_id"].nunique())
print("Log:", TRACK_LOG)


## 5. Seleccionar los 25 tracks más persistentes para las trayectorias

In [ ]:

track_stats = (
    tracks_df.groupby("track_id")
    .agg(
        first_frame=("frame", "min"),
        last_frame=("frame", "max"),
        frames_seen=("frame", "count"),
        mean_confidence=("confidence", "mean"),
    )
    .reset_index()
)

track_stats["span_frames"] = track_stats["last_frame"] - track_stats["first_frame"] + 1
track_stats["span_seconds"] = track_stats["span_frames"] / fps_track
track_stats["coverage"] = track_stats["frames_seen"] / track_stats["span_frames"].clip(lower=1)

# Se prioriza persistencia y continuidad observada.
track_stats = track_stats.sort_values(
    ["span_seconds", "coverage", "mean_confidence"],
    ascending=False
)

TOP_N = 25
highlight_ids = set(track_stats.head(TOP_N)["track_id"].astype(int).tolist())

track_stats.to_csv(RESULTS_OUT / "NB04_estadisticas_tracks.csv", index=False)

print("Tracks resaltados:", sorted(highlight_ids))
print("\nTop 10:")
display(track_stats.head(10))



## 6. Renderizar el video final con trayectorias

Para evitar una visualización saturada:

- se dibujan cajas finas para todos los tracks activos;
- se muestran ID y trayectoria solo en los **25 tracks más persistentes**;
- la trayectoria conserva aproximadamente los últimos 2 segundos.

El panel superior mostrará:
- aves visibles;
- tracks activos;
- tracker empleado;
- nota de que los IDs son temporales.


In [ ]:

FINAL_TEMP = Path("/content/PIO_tracking_trayectorias_temp.mp4")
FINAL_OUT = VIDEOS_OUT / "PIO_tracking_trayectorias_20s.mp4"

cap = cv2.VideoCapture(str(STABLE_CLIP))
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps_render = cap.get(cv2.CAP_PROP_FPS)

writer = cv2.VideoWriter(
    str(FINAL_TEMP),
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps_render,
    (w, h)
)

by_frame = {int(k): g for k, g in tracks_df.groupby("frame")}
history = defaultdict(list)
max_history = max(10, int(2 * fps_render))

frame_idx = 0

while True:
    ok, frame = cap.read()
    if not ok:
        break

    current = by_frame.get(frame_idx, None)

    if current is not None:
        # Todas las detecciones/tracks: cajas discretas
        for _, r in current.iterrows():
            tid = int(r["track_id"])
            x1, y1, x2, y2 = map(int, [r["x1"], r["y1"], r["x2"], r["y2"]])

            is_highlight = tid in highlight_ids
            thickness = 2 if is_highlight else 1
            box_color = (0, 255, 255) if is_highlight else (180, 180, 180)

            cv2.rectangle(frame, (x1, y1), (x2, y2), box_color, thickness)

            cx = int((x1 + x2) / 2)
            cy = int((y1 + y2) / 2)

            if is_highlight:
                history[tid].append((cx, cy))
                if len(history[tid]) > max_history:
                    history[tid] = history[tid][-max_history:]

                cv2.putText(
                    frame, f"ID {tid}",
                    (x1, max(18, y1 - 5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.42,
                    (0, 255, 255), 1, cv2.LINE_AA
                )

                pts = history[tid]
                for p1, p2 in zip(pts[:-1], pts[1:]):
                    cv2.line(frame, p1, p2, (0, 255, 255), 2, cv2.LINE_AA)

        visible_birds = len(current)
        active_tracks = current["track_id"].nunique()
    else:
        visible_birds = 0
        active_tracks = 0

    # Panel superior
    overlay = frame.copy()
    cv2.rectangle(overlay, (20, 20), (680, 174), (0, 0, 0), -1)
    frame = cv2.addWeighted(overlay, 0.58, frame, 0.42, 0)

    cv2.putText(
        frame, "AgroUTMACH PIX | Tracking avicola",
        (38, 58), cv2.FONT_HERSHEY_SIMPLEX, 0.92,
        (255, 255, 255), 2, cv2.LINE_AA
    )
    cv2.putText(
        frame, f"Aves visibles: {visible_birds}    Tracks activos: {active_tracks}",
        (38, 98), cv2.FONT_HERSHEY_SIMPLEX, 0.78,
        (255, 255, 255), 2, cv2.LINE_AA
    )
    cv2.putText(
        frame, f"Tracker: {CHOSEN_TRACKER.replace('.yaml','')}",
        (38, 132), cv2.FONT_HERSHEY_SIMPLEX, 0.65,
        (255, 255, 255), 1, cv2.LINE_AA
    )
    cv2.putText(
        frame, "IDs temporales: no representan identidad biometrica",
        (38, 160), cv2.FONT_HERSHEY_SIMPLEX, 0.52,
        (255, 255, 255), 1, cv2.LINE_AA
    )

    writer.write(frame)
    frame_idx += 1

cap.release()
writer.release()

cmd = (
    f'ffmpeg -y -hide_banner -loglevel error '
    f'-i "{FINAL_TEMP}" -c:v libx264 -preset fast -crf 20 '
    f'-pix_fmt yuv420p -an "{FINAL_OUT}"'
)
rc = os.system(cmd)
assert rc == 0 and FINAL_OUT.exists(), "Falló la codificación final."

print("✓ Video final:", FINAL_OUT)


## 7. Resumen de tracking

In [ ]:

total_unique = int(tracks_df["track_id"].nunique())
duration_tracking = frame_idx / fps_render

summary = {
    "selected_tracker": CHOSEN_TRACKER,
    "video_duration_seconds": float(duration_tracking),
    "unique_temporary_ids_generated": total_unique,
    "highlighted_persistent_tracks": int(len(highlight_ids)),
    "median_track_span_seconds": float(track_stats["span_seconds"].median()),
    "p75_track_span_seconds": float(track_stats["span_seconds"].quantile(0.75)),
    "max_track_span_seconds": float(track_stats["span_seconds"].max()),
    "output_video": str(FINAL_OUT),
}

print(json.dumps(summary, indent=2, ensure_ascii=False))

with open(RESULTS_OUT / "NB04_resumen_tracking.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)


## 8. Gráfica de persistencia de tracks

In [ ]:

plt.figure(figsize=(12, 5))
top_plot = track_stats.head(25).copy()
plt.bar(
    top_plot["track_id"].astype(str),
    top_plot["span_seconds"]
)
plt.xlabel("ID temporal")
plt.ylabel("Persistencia observada (s)")
plt.title("AgroUTMACH PIX — 25 tracks temporales más persistentes")
plt.xticks(rotation=90)
plt.tight_layout()

plot_path = RESULTS_OUT / "NB04_persistencia_top25.png"
plt.savefig(plot_path, dpi=180, bbox_inches="tight")
plt.show()

print("✓ Gráfica:", plot_path)



## 9. Archivos esperados

```text
04_VIDEOS_FERIA/
├── PIO_tracking_estable_original_20s.mp4
└── PIO_tracking_trayectorias_20s.mp4

05_RESULTADOS/NB04_TRACKING/
├── NB04_tramo_estable.json
├── NB04_comparacion_trackers.csv
├── NB04_tracker_seleccionado.json
├── NB04_tracks_por_frame.csv
├── NB04_estadisticas_tracks.csv
├── NB04_resumen_tracking.json
└── NB04_persistencia_top25.png
```

### Siguiente etapa
**NB05 — análisis espacial del galpón: zonas, ocupación y mapa de concentración (heatmap).**


In [ ]:

print("=" * 74)
print("NB04 FINALIZADO CORRECTAMENTE")
print("Tracker:", CHOSEN_TRACKER)
print("Video final:", FINAL_OUT)
print("IDs temporales generados:", total_unique)
print("Tracks resaltados:", len(highlight_ids))
print("=" * 74)
